In [1]:
import torch
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data
import networkx as nx
from torch_geometric.utils import from_networkx
%cd /home/jeroen/repos/traffic-scheduling/network
from automaton import Automaton
from generate import generate_grid_network, generate_simple_instance

/home/jeroen/repos/traffic-scheduling/network


We want to try the message-passing scheme for calculating the earliest starting times (crossing time lower bounds) in the disjunctive graph. This procedure can executed in parallel on the GPU, allowing a batch of disjunctive graphs to be processed at once.

### Single instance

In [140]:
class LowerBounds(MessagePassing):

    def __init__(self, *args):
        super().__init__(aggr='max')
    
    def forward(self, x, weight, edge_index):
        return self.propagate(edge_index, x=x, weight=weight)

    def message(self, x_i, x_j, weight):
        d = torch.max(weight + x_j[:,0], x_i[:,0])
        c = x_j[:,1]
        out = torch.stack((d, c), dim=1)
        return out

In [141]:
G, routes = generate_grid_network(2, 1)
instance = generate_simple_instance(G, routes)
automaton = Automaton(instance)

In [142]:
# add dummy source node
automaton.D.add_node(0, label="source", LB=0, done=0, action_mask=0)
for r in automaton.route_indices:
    v = automaton.route[r][0] # first intersection on route
    automaton.D.add_edge(0, (r, 0, v), weight=0)

# add d and c attributes
nx.set_node_attributes(automaton.D, 0, name="d")
for r, k in automaton.indices:
    v0 = automaton.route[r][0]
    automaton.D.nodes[r, k, v0]['d'] = instance['release'][r][k]

nx.set_node_attributes(automaton.D, 1, name="c")
automaton.D.nodes[0]['c'] = 0

data = from_networkx(automaton.D)

In [145]:
data

Data(edge_index=[2, 129], label=[81], LB=[81], done=[81], action_mask=[81], d=[81], c=[81], weight=[129], num_nodes=81)

In [56]:
lb = LowerBounds()

x = torch.stack((data.d, data.c), dim=1)

while not torch.all(x[:,1] == 0):
    x = lb(x, data.weight, data.edge_index)

In [57]:
d1 = x[:,0]

In [58]:
d2 = torch.tensor(list(nx.get_node_attributes(automaton.D, "LB").values()))

In [59]:
torch.allclose(d1, d2)

True

### Mini-batches

In [109]:
from torch_geometric.loader import DataLoader
import time

N = int(1e4)

G, routes = generate_grid_network(2, 1)
instances = [generate_simple_instance(G, routes) for _ in range(N)]
automata = [Automaton(instance) for instance in instances]

In [112]:
ds1 = []
for automaton in automata:
    ds1.append(torch.tensor(list(nx.get_node_attributes(automaton.D, "LB").values())))

In [110]:
data = []
for instance, automaton in zip(instances, automata):
    # add dummy source node
    automaton.D.add_node(0, label="source", LB=0, done=0, action_mask=0)
    for r in automaton.route_indices:
        v = automaton.route[r][0] # first intersection on route
        automaton.D.add_edge(0, (r, 0, v), weight=0)
    
    # add d and c attributes
    nx.set_node_attributes(automaton.D, 0, name="d")
    for r, k in automaton.indices:
        v0 = automaton.route[r][0]
        automaton.D.nodes[r, k, v0]['d'] = instance['release'][r][k]
    
    nx.set_node_attributes(automaton.D, 1, name="c")
    automaton.D.nodes[0]['c'] = 0
    
    data.append(from_networkx(automaton.D))

In [113]:
class LowerBounds(MessagePassing):

    def __init__(self, *args):
        super().__init__(aggr='max')
    
    def forward(self, x, weight, edge_index):
        return self.propagate(edge_index, x=x, weight=weight)

    def message(self, x_i, x_j, weight):
        d = torch.max(weight + x_j[:,0], x_i[:,0])
        c = x_j[:,1]
        out = torch.stack((d, c), dim=1)
        return out

In [126]:
dataloader = DataLoader(data, batch_size=100, shuffle=False)

In [127]:
from torch_geometric.utils import unbatch

lb = LowerBounds()

start = time.time()
ds2 = []
for batch in dataloader:
    x = torch.stack((batch.d, batch.c), dim=1)
    
    while not torch.all(x[:,1] == 0):
        x = lb(x, batch.weight, batch.edge_index)

    ds2.extend(unbatch(x, batch.batch))

print(time.time() - start)

0.3798832893371582


In [139]:
for d1, d2 in zip(ds1, ds2):
    assert torch.allclose(d1, d2[:,0])